In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import json

In [5]:
PROJECT_DIR = Path.cwd()

# Si le notebook est ouvert depuis Assistant_IA
if PROJECT_DIR.name == "Assistant_IA":
    ASSISTANT_DIR = PROJECT_DIR
else:
    ASSISTANT_DIR = PROJECT_DIR / "Assistant_IA"

DATA_DIR = ASSISTANT_DIR / "data"
ASSISTANT_DB_PATH = DATA_DIR / "citytaste_ottawa.db"
HELP_PATH = DATA_DIR / "site_help.json"

# chemins probables déjà existants dans ton projet principal
MAIN_DATA_DIR = ASSISTANT_DIR.parent / "data"
PROCESSED_DIR = MAIN_DATA_DIR / "processed"

CSV_PATH = PROCESSED_DIR / "ottawa_places_enriched_google_photos.csv"
REAL_DB_PATH = PROCESSED_DIR / "citytaste_ottawa.db"

print("ASSISTANT_DIR =", ASSISTANT_DIR)
print("ASSISTANT_DB_PATH =", ASSISTANT_DB_PATH)
print("HELP_PATH =", HELP_PATH)
print("CSV_PATH exists ?", CSV_PATH.exists(), "->", CSV_PATH)
print("REAL_DB_PATH exists ?", REAL_DB_PATH.exists(), "->", REAL_DB_PATH)

ASSISTANT_DIR = c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA
ASSISTANT_DB_PATH = c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA\data\citytaste_ottawa.db
HELP_PATH = c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA\data\site_help.json
CSV_PATH exists ? True -> c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\data\processed\ottawa_places_enriched_google_photos.csv
REAL_DB_PATH exists ? True -> c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\data\processed\citytaste_ottawa.db


In [6]:
DB_PATH = REAL_DB_PATH
print("On utilise la vraie base :", DB_PATH)

On utilise la vraie base : c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\data\processed\citytaste_ottawa.db


In [7]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

(1104, 34)


,osm_type,osm_id,name,place_type,amenity,tourism,cuisine,lat,lon,addr_housenumber,...,eval_label,google_rating,google_user_rating_count,rating_source,google_match_score,google_place_id,google_photo_name,google_photo_url,google_photo_attribution,google_photo_match_score
0,relation,14526702,Les Suites Hotel,hotel,Unknown,hotel,Unknown,45.426134,-75.688731,130,...,hotel,4.0,2249.0,google_places_api,0.8637,ChIJpXHm3wMFzkwRLC22CEvfRYg,places/ChIJpXHm3wMFzkwRLC22CEvfRYg/photos/ATCD...,https://places.googleapis.com/v1/places/ChIJpX...,Les Suites Hotel Ottawa,0.8637
1,way,31813259,Embassy Hotel and Suites,hotel,Unknown,hotel,Unknown,45.419679,-75.688818,25,...,hotel,4.0,1277.0,google_places_api,0.8933,ChIJJSkEpKkFzkwRgKmEk0gQEA8,places/ChIJJSkEpKkFzkwRgKmEk0gQEA8/photos/ATCD...,https://places.googleapis.com/v1/places/ChIJJS...,Ottawa Embassy Hotel and Suites,0.8933
2,way,31813278,The Business Inn & Suites,hotel,Unknown,hotel,Unknown,45.416980,-75.689237,180,...,hotel,4.4,2778.0,google_places_api,0.9429,ChIJOViXzBYPzkwRwgwgubhv0Lk,places/ChIJOViXzBYPzkwRwgwgubhv0Lk/photos/ATCD...,https://places.googleapis.com/v1/places/ChIJOV...,The Business Inn & Suites,0.9429
3,way,46016942,Ottawa Jail Hostel,hostel,Unknown,hostel,Unknown,45.425111,-75.688451,75,...,hostel,4.3,2395.0,google_places_api,0.8394,ChIJQ4GyNgEFzkwRdukc-ESL2IA,places/ChIJQ4GyNgEFzkwRdukc-ESL2IA/photos/ATCD...,https://places.googleapis.com/v1/places/ChIJQ4...,Saintlo Ottawa Jail Hostel,0.8394
4,way,55411730,Sonder Rideau Apartments Downtown,hotel,Unknown,hotel,Unknown,45.420741,-75.694605,161,...,hotel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df.columns.tolist()

['osm_type',
 'osm_id',
 'name',
 'place_type',
 'amenity',
 'tourism',
 'cuisine',
 'lat',
 'lon',
 'addr_housenumber',
 'addr_street',
 'addr_city',
 'addr_postcode',
 'address',
 'phone',
 'website',
 'opening_hours',
 'wheelchair',
 'brand',
 'source',
 'tags_json',
 'address_clean',
 'dist_to_center_km',
 'text',
 'eval_label',
 'google_rating',
 'google_user_rating_count',
 'rating_source',
 'google_match_score',
 'google_place_id',
 'google_photo_name',
 'google_photo_url',
 'google_photo_attribution',
 'google_photo_match_score']

### Creation de la table place dans la base assistant

In [9]:
with sqlite3.connect(ASSISTANT_DB_PATH) as conn:
    df.to_sql("places", conn, if_exists="replace", index=False)

DB_PATH = ASSISTANT_DB_PATH
print("Base assistant créée :", DB_PATH)

Base assistant créée : c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA\data\citytaste_ottawa.db


In [10]:
# Inspection de la base de données
with sqlite3.connect(DB_PATH) as conn:
    tables_df = pd.read_sql_query("""
        SELECT name
        FROM sqlite_master
        WHERE type='table'
        ORDER BY name
    """, conn)

tables_df

,name
0,places


In [16]:
# Voir les colonnes de la table places
def get_table_columns(table_name: str):
    with sqlite3.connect(DB_PATH) as conn:
        info_df = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)
    return info_df[["name", "type"]]

get_table_columns("places")

,name,type
0,osm_type,TEXT
1,osm_id,INTEGER
2,name,TEXT
3,place_type,TEXT
4,amenity,TEXT
5,tourism,TEXT
6,cuisine,TEXT
7,lat,REAL
8,lon,REAL
9,addr_housenumber,TEXT


In [ ]:
# Quelque lignes
with sqlite3.connect(DB_PATH) as conn:
    preview_df = pd.read_sql_query("SELECT * FROM places LIMIT 5", conn)

preview_df

,osm_type,osm_id,name,place_type,amenity,tourism,cuisine,lat,lon,addr_housenumber,...,eval_label,google_rating,google_user_rating_count,rating_source,google_match_score,google_place_id,google_photo_name,google_photo_url,google_photo_attribution,google_photo_match_score
0,relation,14526702,Les Suites Hotel,hotel,Unknown,hotel,Unknown,45.426134,-75.688731,130,...,hotel,4.0,2249.0,google_places_api,0.8637,ChIJpXHm3wMFzkwRLC22CEvfRYg,places/ChIJpXHm3wMFzkwRLC22CEvfRYg/photos/ATCD...,https://places.googleapis.com/v1/places/ChIJpX...,Les Suites Hotel Ottawa,0.8637
1,way,31813259,Embassy Hotel and Suites,hotel,Unknown,hotel,Unknown,45.419679,-75.688818,25,...,hotel,4.0,1277.0,google_places_api,0.8933,ChIJJSkEpKkFzkwRgKmEk0gQEA8,places/ChIJJSkEpKkFzkwRgKmEk0gQEA8/photos/ATCD...,https://places.googleapis.com/v1/places/ChIJJS...,Ottawa Embassy Hotel and Suites,0.8933
2,way,31813278,The Business Inn & Suites,hotel,Unknown,hotel,Unknown,45.416980,-75.689237,180,...,hotel,4.4,2778.0,google_places_api,0.9429,ChIJOViXzBYPzkwRwgwgubhv0Lk,places/ChIJOViXzBYPzkwRwgwgubhv0Lk/photos/ATCD...,https://places.googleapis.com/v1/places/ChIJOV...,The Business Inn & Suites,0.9429
3,way,46016942,Ottawa Jail Hostel,hostel,Unknown,hostel,Unknown,45.425111,-75.688451,75,...,hostel,4.3,2395.0,google_places_api,0.8394,ChIJQ4GyNgEFzkwRdukc-ESL2IA,places/ChIJQ4GyNgEFzkwRdukc-ESL2IA/photos/ATCD...,https://places.googleapis.com/v1/places/ChIJQ4...,Saintlo Ottawa Jail Hostel,0.8394
4,way,55411730,Sonder Rideau Apartments Downtown,hotel,Unknown,hotel,Unknown,45.420741,-75.694605,161,...,hotel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Lecture du JSON

In [17]:
with open(HELP_PATH, "r", encoding="utf-8") as f:
    site_help = json.load(f)

site_help

[{'question': 'Comment chercher un lieu ?',
  'answer': 'Utilisez les filtres disponibles sur la page de recherche, puis lancez la recommandation pour afficher les résultats.'},
 {'question': 'Que signifie la distance ?',
  'answer': "La distance correspond à une référence de localisation utilisée par l'application. Si la position réelle de l'utilisateur n'est pas activée, la distance n'est pas forcément calculée depuis sa position."},
 {'question': "Comment voir les détails d'un lieu ?",
  'answer': 'Cliquez sur le résultat ou sur la carte du lieu pour ouvrir sa fiche détaillée.'},
 {'question': "Pourquoi certains lieux n'ont pas d'image ?",
  'answer': "Certaines données sources ne fournissent pas d'image. Dans ce cas, un visuel de remplacement peut être affiché."}]

In [18]:
# Vérification de quelques colonnes utiles
with sqlite3.connect(DB_PATH) as conn:
    preview_df = pd.read_sql_query("""
        SELECT
            rowid AS internal_id,
            name,
            place_type,
            cuisine,
            address,
            dist_to_center_km,
            google_rating,
            google_user_rating_count
        FROM places
        LIMIT 10
    """, conn)

preview_df

,internal_id,name,place_type,cuisine,address,dist_to_center_km,google_rating,google_user_rating_count
0,1,Les Suites Hotel,hotel,Unknown,130 Besserer Street,0.838080,4.0,2249.0
1,2,Embassy Hotel and Suites,hotel,Unknown,25 Cartier Street,0.684773,4.0,1277.0
2,3,The Business Inn & Suites,hotel,Unknown,180 MacLaren Street,0.799298,4.4,2778.0
3,4,Ottawa Jail Hostel,hostel,Unknown,75 Nicholas Street,0.792124,4.3,2395.0
4,5,Sonder Rideau Apartments Downtown,hotel,Unknown,161 Laurier Avenue West,0.219363,NaN,NaN
5,6,Hampton Inn by Hilton Ottawa,hotel,Unknown,100 Coventry Road,3.134404,4.4,2321.0
6,7,Perkins,restaurant,american,1130 St-Laurent Boulevard,4.605865,NaN,NaN
7,8,Capital Hill Hotel & Suites,hotel,Unknown,88 Albert Street,0.125813,NaN,NaN
8,9,St-Hubert,restaurant,chicken;barbecue,1754 St-Laurent Boulevard,5.771821,NaN,NaN
9,10,Senpai,restaurant,sushi,2280 Carling Avenue,8.465806,NaN,NaN


In [20]:
# Creation de la fonction search_places
def search_places(
    place_type=None,
    cuisine=None,
    max_distance_km=None,
    min_rating=None,
    keyword=None,
    limit=5
):
    sql = """
    SELECT
        rowid AS internal_id,
        osm_type,
        osm_id,
        name,
        place_type,
        amenity,
        tourism,
        cuisine,
        address,
        addr_city,
        opening_hours,
        wheelchair,
        website,
        google_rating,
        google_user_rating_count,
        google_photo_url,
        dist_to_center_km,
        text
    FROM places
    WHERE 1=1
    """
    
    params = []

    if place_type:
        sql += " AND LOWER(COALESCE(place_type, '')) = LOWER(?)"
        params.append(place_type)

    if cuisine:
        sql += " AND LOWER(COALESCE(cuisine, '')) LIKE LOWER(?)"
        params.append(f"%{cuisine}%")

    if max_distance_km is not None:
        sql += " AND dist_to_center_km IS NOT NULL AND dist_to_center_km <= ?"
        params.append(max_distance_km)

    if min_rating is not None:
        sql += " AND google_rating IS NOT NULL AND google_rating >= ?"
        params.append(min_rating)

    if keyword:
        sql += """
        AND (
            LOWER(COALESCE(name, '')) LIKE LOWER(?)
            OR LOWER(COALESCE(cuisine, '')) LIKE LOWER(?)
            OR LOWER(COALESCE(address, '')) LIKE LOWER(?)
            OR LOWER(COALESCE(text, '')) LIKE LOWER(?)
            OR LOWER(COALESCE(amenity, '')) LIKE LOWER(?)
            OR LOWER(COALESCE(tourism, '')) LIKE LOWER(?)
        )
        """
        kw = f"%{keyword}%"
        params.extend([kw, kw, kw, kw, kw, kw])

    sql += """
    ORDER BY
        CASE WHEN google_rating IS NULL THEN 1 ELSE 0 END,
        google_rating DESC,
        CASE WHEN dist_to_center_km IS NULL THEN 1 ELSE 0 END,
        dist_to_center_km ASC
    LIMIT ?
    """
    params.append(limit)

    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(sql, conn, params=params)

    return df

In [ ]:
# Test 1
# Tester search_places
search_places(limit=5)

,internal_id,osm_type,osm_id,name,place_type,amenity,tourism,cuisine,address,addr_city,opening_hours,wheelchair,website,google_rating,google_user_rating_count,google_photo_url,dist_to_center_km,text
0,1099,node,13334898201,Ibex Habesha Restaurant,restaurant,restaurant,Unknown,african;ethiopian,164 Laurier Avenue West,Ottawa,Mo-Su 09:30-21:00,Unknown,https://ibexhabesha.ca/,5.0,384.0,https://places.googleapis.com/v1/places/ChIJF2...,0.253738,Ibex Habesha Restaurant restaurant african;eth...
1,459,way,585009255,Belle Notte Bed&Breakfast,guest_house,Unknown,guest_house,Unknown,108 Daly Avenue,Ottawa,NaN,Unknown,NaN,5.0,25.0,https://places.googleapis.com/v1/places/ChIJeb...,1.075795,Belle Notte Bed&Breakfast guest_house Unknown ...
2,53,way,161485771,Macintosh's CiderHouse,restaurant,restaurant,Unknown,Unknown,NaN,Ottawa,NaN,Unknown,NaN,5.0,10.0,https://places.googleapis.com/v1/places/ChIJP8...,34.891268,Macintosh's CiderHouse restaurant Unknown nan ...
3,570,way,700636237,The Stewart Bed and Breakfast,guest_house,Unknown,guest_house,Unknown,74 Stewart Street,Ottawa,NaN,Unknown,https://www.thestewartbedandbreakfastottawa.com,4.9,107.0,https://places.googleapis.com/v1/places/ChIJ7Q...,1.087238,The Stewart Bed and Breakfast guest_house Unkn...
4,113,way,255396635,Little Amsterdam,restaurant,restaurant,Unknown,dutch,190 Dalhousie Street,Ottawa,NaN,Unknown,NaN,4.9,33.0,https://places.googleapis.com/v1/places/ChIJo4...,1.209494,Little Amsterdam restaurant dutch 190 Dalhousi...


In [22]:
# Test 2
search_places(place_type="restaurant", limit=5)

,internal_id,osm_type,osm_id,name,place_type,amenity,tourism,cuisine,address,addr_city,opening_hours,wheelchair,website,google_rating,google_user_rating_count,google_photo_url,dist_to_center_km,text
0,1099,node,13334898201,Ibex Habesha Restaurant,restaurant,restaurant,Unknown,african;ethiopian,164 Laurier Avenue West,Ottawa,Mo-Su 09:30-21:00,Unknown,https://ibexhabesha.ca/,5.0,384.0,https://places.googleapis.com/v1/places/ChIJF2...,0.253738,Ibex Habesha Restaurant restaurant african;eth...
1,53,way,161485771,Macintosh's CiderHouse,restaurant,restaurant,Unknown,Unknown,NaN,Ottawa,NaN,Unknown,NaN,5.0,10.0,https://places.googleapis.com/v1/places/ChIJP8...,34.891268,Macintosh's CiderHouse restaurant Unknown nan ...
2,113,way,255396635,Little Amsterdam,restaurant,restaurant,Unknown,dutch,190 Dalhousie Street,Ottawa,NaN,Unknown,NaN,4.9,33.0,https://places.googleapis.com/v1/places/ChIJo4...,1.209494,Little Amsterdam restaurant dutch 190 Dalhousi...
3,1047,node,10708126186,Fragrant Noodles,restaurant,restaurant,Unknown,chinese;noodle,358 Rideau Street,Ottawa,NaN,Unknown,NaN,4.9,127.0,https://places.googleapis.com/v1/places/ChIJyV...,1.393775,Fragrant Noodles restaurant chinese;noodle 358...
4,512,way,607449917,Govinda's,restaurant,restaurant,Unknown,indian,212 Somerset Street East,Ottawa,Mo-Fr 17:00-20:00,Unknown,https://www.ottawa.iskcon.ca/homepage/buffet2.htm,4.9,312.0,https://places.googleapis.com/v1/places/ChIJZW...,1.430097,Govinda's restaurant indian 212 Somerset Stree...


In [23]:
# Test 3
search_places(place_type="restaurant", cuisine="italian", limit=5)

,internal_id,osm_type,osm_id,name,place_type,amenity,tourism,cuisine,address,addr_city,opening_hours,wheelchair,website,google_rating,google_user_rating_count,google_photo_url,dist_to_center_km,text
0,632,node,2295959718,Sanguiccio,restaurant,restaurant,Unknown,italian,183 Preston Street,Ottawa,NaN,no,https://www.sanguiccio.ca/,4.9,230.0,https://places.googleapis.com/v1/places/ChIJd6...,2.068228,Sanguiccio restaurant italian 183 Preston Stre...
1,692,node,3522632487,Brassica,restaurant,restaurant,Unknown,italian,309 Richmond Road,Ottawa,NaN,no,NaN,4.8,226.0,https://places.googleapis.com/v1/places/ChIJn8...,5.376428,"Brassica restaurant italian 309 Richmond Road,..."
2,896,node,5836839082,Ciao Italia,restaurant,restaurant,Unknown,italian,641 Somerset Street West,Ottawa,NaN,Unknown,NaN,4.7,597.0,https://places.googleapis.com/v1/places/ChIJOY...,1.220281,Ciao Italia restaurant italian 641 Somerset St...
3,731,node,4243547893,Retro Gusto,restaurant,restaurant,Unknown,italian,122 Preston Street,Ottawa,Mo-Su 17:00-22:00,no,https://www.retrogustoeats.com/,4.7,369.0,https://places.googleapis.com/v1/places/ChIJ_c...,1.977196,Retro Gusto restaurant italian 122 Preston Str...
4,486,way,597564047,Cucina da Vito,restaurant,restaurant,Unknown,italian,2701 St. Joseph Boulevard,Ottawa,NaN,Unknown,NaN,4.7,727.0,https://places.googleapis.com/v1/places/ChIJ-Q...,14.664136,Cucina da Vito restaurant italian 2701 St. Jos...


In [24]:
# Test 4
search_places(place_type="restaurant", max_distance_km=5, limit=5)

,internal_id,osm_type,osm_id,name,place_type,amenity,tourism,cuisine,address,addr_city,opening_hours,wheelchair,website,google_rating,google_user_rating_count,google_photo_url,dist_to_center_km,text
0,1099,node,13334898201,Ibex Habesha Restaurant,restaurant,restaurant,Unknown,african;ethiopian,164 Laurier Avenue West,Ottawa,Mo-Su 09:30-21:00,Unknown,https://ibexhabesha.ca/,5.0,384.0,https://places.googleapis.com/v1/places/ChIJF2...,0.253738,Ibex Habesha Restaurant restaurant african;eth...
1,113,way,255396635,Little Amsterdam,restaurant,restaurant,Unknown,dutch,190 Dalhousie Street,Ottawa,NaN,Unknown,NaN,4.9,33.0,https://places.googleapis.com/v1/places/ChIJo4...,1.209494,Little Amsterdam restaurant dutch 190 Dalhousi...
2,1047,node,10708126186,Fragrant Noodles,restaurant,restaurant,Unknown,chinese;noodle,358 Rideau Street,Ottawa,NaN,Unknown,NaN,4.9,127.0,https://places.googleapis.com/v1/places/ChIJyV...,1.393775,Fragrant Noodles restaurant chinese;noodle 358...
3,512,way,607449917,Govinda's,restaurant,restaurant,Unknown,indian,212 Somerset Street East,Ottawa,Mo-Fr 17:00-20:00,Unknown,https://www.ottawa.iskcon.ca/homepage/buffet2.htm,4.9,312.0,https://places.googleapis.com/v1/places/ChIJZW...,1.430097,Govinda's restaurant indian 212 Somerset Stree...
4,273,way,479521654,Dinette Atomique,restaurant,restaurant,Unknown,Unknown,321 Somerset Street East,Ottawa,Mo-Sa 09:00-22:00 Su 09:00-16:00,Unknown,https://dinetteatomique.ca/,4.9,142.0,https://places.googleapis.com/v1/places/ChIJPf...,1.824492,Dinette Atomique restaurant Unknown 321 Somers...


In [25]:
# Test 5
search_places(keyword="hotel", limit=5)

,internal_id,osm_type,osm_id,name,place_type,amenity,tourism,cuisine,address,addr_city,opening_hours,wheelchair,website,google_rating,google_user_rating_count,google_photo_url,dist_to_center_km,text
0,302,way,482458497,Adam's Airport Inn,hotel,Unknown,hotel,Unknown,2721 Bank Street,Ottawa,None,Unknown,NaN,4.7,926.0,https://places.googleapis.com/v1/places/ChIJPw...,9.179669,Adam's Airport Inn hotel Unknown 2721 Bank Str...
1,901,node,5839731606,Le Germain Hotel Ottawa,hotel,Unknown,hotel,Unknown,30 Daly Avenue,Ottawa,None,Unknown,https://www.legermainhotels.com/en/ottawa/contact,4.6,784.0,https://places.googleapis.com/v1/places/ChIJ1T...,0.853673,Le Germain Hotel Ottawa hotel Unknown 30 Daly ...
2,136,way,292813447,McGee's Inn,guest_house,Unknown,guest_house,Unknown,185 Daly Avenue,Ottawa,None,Unknown,NaN,4.6,192.0,https://places.googleapis.com/v1/places/ChIJQ5...,1.321579,McGee's Inn guest_house Unknown 185 Daly Avenu...
3,567,way,683258836,Fairfield Inn & Suites Ottawa Airport,hotel,Unknown,hotel,Unknown,135 Thad Johnson Private,Ottawa,None,Unknown,NaN,4.6,1328.0,https://places.googleapis.com/v1/places/ChIJce...,10.974101,Fairfield Inn & Suites Ottawa Airport hotel Un...
4,545,way,658477194,Homewood Suites Ottawa Kanata,hotel,Unknown,hotel,Unknown,900 Great Lakes Avenue,Ottawa,None,Unknown,NaN,4.6,746.0,https://places.googleapis.com/v1/places/ChIJl7...,19.958101,Homewood Suites Ottawa Kanata hotel Unknown 90...


### Creation de la fonction get_place_details()


In [26]:
def get_place_details(internal_id):
    sql = """
    SELECT
        rowid AS internal_id,
        *
    FROM places
    WHERE rowid = ?
    """

    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(sql, conn, params=[internal_id])

    if df.empty:
        return None

    return df.iloc[0].to_dict()

In [27]:
# Tester la fonction
get_place_details(1)

{'internal_id': 1,
 'osm_type': 'relation',
 'osm_id': 14526702,
 'name': 'Les Suites Hotel',
 'place_type': 'hotel',
 'amenity': 'Unknown',
 'tourism': 'hotel',
 'cuisine': 'Unknown',
 'lat': 45.4261338,
 'lon': -75.6887306,
 'addr_housenumber': '130',
 'addr_street': 'Besserer Street',
 'addr_city': 'Ottawa',
 'addr_postcode': 'K1N 9M9',
 'address': '130 Besserer Street',
 'phone': '+1 613 232 2000',
 'website': 'https://www.les-suites.com/',
 'opening_hours': None,
 'wheelchair': 'yes',
 'brand': 'Unknown',
 'source': 'City of Ottawa',
 'tags_json': '{"@id": "relation/14526702", "addr:city": "Ottawa", "addr:housenumber": "130", "addr:postcode": "K1N 9M9", "addr:province": "ON", "addr:street": "Besserer Street", "air_conditioning": "yes", "building": "hotel", "email": "Reachout@les-suites.com", "fax": "+1 613 232 1242", "internet_access": "yes", "internet_access:fee": "customers", "internet_access:ssid": "LesSuites_Guest", "name": "Les Suites Hotel", "phone": "+1 613 232 2000", "sour

### Creation des fonctions pour l'aide du site

In [29]:
import re
import unicodedata

def normalize_text(text):
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_site_help():
    with open(HELP_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def answer_site_help(user_question):
    user_norm = normalize_text(user_question)
    user_tokens = set(user_norm.split())

    site_help = load_site_help()

    best_item = None
    best_score = -1

    for item in site_help:
        q_norm = normalize_text(item["question"])
        q_tokens = set(q_norm.split())

        overlap = len(user_tokens & q_tokens)

        if q_norm in user_norm:
            overlap += 2

        if overlap > best_score:
            best_score = overlap
            best_item = item

    if best_item is None or best_score <= 0:
        return "Je n’ai pas encore de réponse précise pour cette question sur le site."

    return best_item["answer"]

In [30]:
# Tester l'aide du site
answer_site_help("Comment chercher un lieu ?")

'Utilisez les filtres disponibles sur la page de recherche, puis lancez la recommandation pour afficher les résultats.'

In [31]:
answer_site_help("Comment voir les détails ?")

'Cliquez sur le résultat ou sur la carte du lieu pour ouvrir sa fiche détaillée.'

In [32]:
answer_site_help("La distance veut dire quoi ?")

"La distance correspond à une référence de localisation utilisée par l'application. Si la position réelle de l'utilisateur n'est pas activée, la distance n'est pas forcément calculée depuis sa position."

### Creation d'un mini routeur locale

In [33]:
def is_site_question(user_message):
    text = normalize_text(user_message)

    site_keywords = [
        "site", "page", "interface", "filtre", "filtres", "recherche",
        "distance", "image", "images", "detail", "details",
        "comment", "fonctionne", "utiliser"
    ]

    return any(word in text for word in site_keywords)


def simple_citytaste_assistant(user_message):
    if is_site_question(user_message):
        return {
            "type": "site_help",
            "answer": answer_site_help(user_message)
        }

    # exemples très simples de logique côté lieux
    text = normalize_text(user_message)

    place_type = None
    cuisine = None

    if "restaurant" in text or "resto" in text:
        place_type = "restaurant"
    elif "hotel" in text or "hebergement" in text or "accommodation" in text:
        place_type = "accommodation"

    known_cuisines = [
        "italian", "indian", "chinese", "pizza", "vietnamese", "african", "lebanese"
    ]

    for c in known_cuisines:
        if c in text:
            cuisine = c
            break

    results = search_places(
        place_type=place_type,
        cuisine=cuisine,
        limit=3
    )

    if results.empty:
        return {
            "type": "places",
            "answer": "Je n’ai pas trouvé de lieu correspondant pour le moment."
        }

    return {
        "type": "places",
        "answer": results[[
            "internal_id", "name", "place_type", "cuisine",
            "address", "dist_to_center_km", "google_rating"
        ]].to_dict(orient="records")
    }

In [ ]:
# Tester le mini assistant
simple_citytaste_assistant("Comment chercher un lieu ?")

{'type': 'site_help',
 'answer': 'Utilisez les filtres disponibles sur la page de recherche, puis lancez la recommandation pour afficher les résultats.'}

In [38]:
simple_citytaste_assistant("Je cherche un restaurant italien proche du centre avec une bonne note.")

{'type': 'places',
 'answer': [{'internal_id': 1099,
   'name': 'Ibex Habesha Restaurant',
   'place_type': 'restaurant',
   'cuisine': 'african;ethiopian',
   'address': '164 Laurier Avenue West',
   'dist_to_center_km': 0.2537384200394798,
   'google_rating': 5.0},
  {'internal_id': 53,
   'name': "Macintosh's CiderHouse",
   'place_type': 'restaurant',
   'cuisine': 'Unknown',
   'address': nan,
   'dist_to_center_km': 34.89126784197973,
   'google_rating': 5.0},
  {'internal_id': 113,
   'name': 'Little Amsterdam',
   'place_type': 'restaurant',
   'cuisine': 'dutch',
   'address': '190 Dalhousie Street',
   'dist_to_center_km': 1.2094939669680589,
   'google_rating': 4.9}]}

In [39]:
simple_citytaste_assistant("Montre-moi un hotel")

{'type': 'places',
 'answer': 'Je n’ai pas trouvé de lieu correspondant pour le moment.'}

### Test utile avant lancement

In [42]:
from pathlib import Path
import sqlite3
import pandas as pd

BASE_DIR = Path.cwd()

# si le notebook est déjà ouvert dans Assistant_IA
if BASE_DIR.name == "Assistant_IA":
    DB_PATH = BASE_DIR / "data" / "citytaste_ottawa.db"
else:
    DB_PATH = BASE_DIR / "Assistant_IA" / "data" / "citytaste_ottawa.db"

print("Current working dir:", BASE_DIR)
print("DB_PATH:", DB_PATH)
print("Exists?", DB_PATH.exists())
with sqlite3.connect(DB_PATH) as conn:
    print(pd.read_sql_query("""
        SELECT place_type, COUNT(*) as n
        FROM places
        GROUP BY place_type
        ORDER BY n DESC
    """, conn))

    print(pd.read_sql_query("""
        SELECT tourism, COUNT(*) as n
        FROM places
        WHERE tourism IS NOT NULL AND TRIM(tourism) <> ''
        GROUP BY tourism
        ORDER BY n DESC
        LIMIT 20
    """, conn))

    print(pd.read_sql_query("""
        SELECT amenity, COUNT(*) as n
        FROM places
        WHERE amenity IS NOT NULL AND TRIM(amenity) <> ''
        GROUP BY amenity
        ORDER BY n DESC
        LIMIT 20
    """, conn))

Current working dir: c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA
DB_PATH: c:\Users\Innocent\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Projet Capstone en IA\CityTaste\Assistant_IA\data\citytaste_ottawa.db
Exists? True
    place_type     n
0   restaurant  1009
1        hotel    66
2  guest_house    16
3        motel    10
4       hostel     3
       tourism     n
0      Unknown  1009
1        hotel    66
2  guest_house    16
3        motel    10
4       hostel     3
      amenity     n
0  restaurant  1009
1     Unknown    95
